# Capstone: Pedestrian AEB from Requirement to Release Decision

**Module 08 · Standards, Regulation and Safety Case** · Notebook 5 of 5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/Autonomous-Driving_AI-Safety-and-Security/blob/main/08_Standards_Regulation_and_Safety_Case/05_capstone_aeb_from_requirement_to_release.ipynb)

| Level | Prerequisites | Standards |
|---|---|---|
| Advanced | [HARA](../03_Functional_Safety_ISO26262/02_hara.ipynb), [TARA](../05_Cybersecurity_ISO21434/02_tara.ipynb), [Calibration](../06_AI_Safety_ISO_PAS_8800/04_calibration.ipynb), [Safety case with GSN](04_safety_case_gsn.ipynb) | ISO 26262:2018, ISO 21448:2022, ISO/SAE 21434:2021, ISO/PAS 8800:2024, UL 4600 (3rd ed.) |

**Author:** Milin Patel · Hochschule Kempten · MIT Licence

## Learning objectives

By the end of this notebook you will be able to:

1. Trace one function, pedestrian automated emergency braking (AEB), from item definition and HARA through the functional safety concept to a release decision, reusing the course's work products instead of re-deriving them.
2. Derive a quantitative perception requirement from vehicle dynamics and verify it on **real KITTI images with a real detector**, including recall by distance and occlusion.
3. Turn measured performance insufficiencies into SOTIF measures: an ODD restriction computed from the data, and triggering conditions handed to the validation plan.
4. Use calibration and split conformal prediction as evidence, and state what each does and does not guarantee.
5. Combine the TARA, the validation target and scenario coverage into a GSN-shaped evidence table.
6. Compute a release checklist and decision (*release / release with ODD restriction / do not release*), and explain why the evidence in a notebook can never be enough on its own.

## How this notebook is organised

Each section produces one work product of a real project. Theory is not repeated: every step links to the notebook where the method is taught.

| § | Work product | Standard (clause) | Taught in |
|---|---|---|---|
| 1 | Item definition and ODD | ISO 26262-3 cl. 5; ISO 34503 | [03-02 HARA](../03_Functional_Safety_ISO26262/02_hara.ipynb), [01-01 SAE levels and ODD](../01_Perception/01_sae_levels_and_odd.ipynb) |
| 2 | HARA summary, safety goals | ISO 26262-3 cl. 6 | [03-02 HARA](../03_Functional_Safety_ISO26262/02_hara.ipynb) |
| 3 | Functional safety concept, ASIL decomposition, FTA check | ISO 26262-3 cl. 7; -9 cl. 5, 7 | [03-04 FTA](../03_Functional_Safety_ISO26262/04_fault_tree_analysis.ipynb), [03-03 FMEDA](../03_Functional_Safety_ISO26262/03_fmea_and_fmeda.ipynb) |
| 4 | Perception performance requirement and its verification on KITTI | ISO 21448 (triggering conditions, acceptance criteria) | [01-05 VRU detection](../01_Perception/05_vulnerable_road_user_detection.ipynb), [04-02 Triggering conditions](../04_SOTIF_ISO21448/02_triggering_conditions_and_scenarios.ipynb) |
| 5 | Uncertainty evidence: calibration, conformal recall threshold | ISO/PAS 8800 | [06-04 Calibration](../06_AI_Safety_ISO_PAS_8800/04_calibration.ipynb), [06-05 Conformal prediction](../06_AI_Safety_ISO_PAS_8800/05_conformal_prediction.ipynb) |
| 6 | TARA summary and CAL for the perception → brake path | ISO/SAE 21434 cl. 15, 9.4 | [05-02 TARA](../05_Cybersecurity_ISO21434/02_tara.ipynb) |
| 7 | Validation target and scenario coverage | ISO 21448 cl. 10–11; ISO 34502 | [06-08 Statistical validation](../06_AI_Safety_ISO_PAS_8800/08_statistical_safety_validation.ipynb), [04-04 Simulation-based validation](../04_SOTIF_ISO21448/04_simulation_based_validation.ipynb) |
| 8 | Safety case summary (GSN) | UL 4600; ISO 26262-2 (safety case) | [08-04 Safety case with GSN](04_safety_case_gsn.ipynb) |
| 9 | Release decision | ISO 26262-2 (assessment, release); ISO 21448 cl. 12 | this notebook |

> **Illustrative data.** Ratings, failure rates, timings and policy thresholds are teaching values, marked where they appear. The perception evidence in §4–§5 is **measured** on real KITTI images.

In [ ]:
# Course toolkit (tables, formulas and real-data loaders). Installs automatically on Colab.
try:
    import avsafety
except ImportError:
    %pip install -q "avsafety[data] @ git+https://github.com/milinpatel07/Autonomous-Driving_AI-Safety-and-Security.git"
    import avsafety
from avsafety import data as D
from avsafety import coverage as COV, iso21434 as I21434, iso26262 as I26262, sotif as SOTIF, uq as UQ, validation as VAL

import itertools
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import Markdown, display

SEED = 2026
rng = np.random.default_rng(SEED)
OI = {"blue": "#0072B2", "orange": "#E69F00", "green": "#009E73", "vermillion": "#D55E00",
      "sky": "#56B4E9", "purple": "#CC79A7", "yellow": "#F0E442", "grey": "#999999"}   # Okabe-Ito
pd.set_option("display.max_colwidth", 90)
print(f"avsafety {avsafety.__version__}; seed {SEED}")

### Data and licence

The perception evidence uses 40 frames of the **KITTI object detection benchmark** (left colour camera, Karlsruhe, daytime, dry weather), read one file at a time from the official archive. KITTI is licensed **CC BY-NC-SA 3.0** (non-commercial use; cite Geiger et al. 2012). The detector is torchvision's `fasterrcnn_mobilenet_v3_large_fpn` with weights pretrained on **COCO**; it has never seen KITTI, which is the honest situation for a supplier component evaluated on the integrator's data.

In [ ]:
print(D.attribution("KITTI"))
print("Detector: torchvision fasterrcnn_mobilenet_v3_large_fpn, COCO-pretrained weights (download.pytorch.org)")

## 1. Item definition and ODD

The item is the AEB function of [03-02](../03_Functional_Safety_ISO26262/02_hara.ipynb): camera and radar perception, threat assessment and the brake request, active at **10–130 km/h**. AEB is a momentary-intervention active safety function and lies outside the SAE J3016 levels; the driver remains responsible. This capstone follows the **pedestrian sub-scope**: urban roads up to **60 km/h**.

The ODD table below is the *specification*. Column "Evidence in §4" is filled from the data later, so that the gap between what is specified and what is evidenced stays visible.

In [ ]:
ITEM = {
    "Item": "Automated emergency braking (AEB), passenger car",
    "Function": "Detect road users in the path; if a collision is imminent and the driver does not react, request up to full braking",
    "Boundary": "Front camera, front radar, perception and fusion, threat assessment, brake request (brake system = neighbouring item)",
    "Operating range": "Active 10-130 km/h; pedestrian sub-scope: urban, <= 60 km/h",
    "Driver role": "Responsible at all times (active safety, outside SAE J3016 levels)",
}
ODD = pd.DataFrame([
    ("Road type", "urban, 2-4 lanes, with sidewalks", "urban and residential streets (Karlsruhe)"),
    ("Ego speed", "10-60 km/h (pedestrian sub-scope)", "not observable in single frames"),
    ("Lighting", "day, dusk, night with street lighting", "day only"),
    ("Weather", "dry, rain", "dry only"),
    ("Pedestrians", "adults and children, standing, walking, crossing", "adults and children (measured below)"),
    ("Occlusion", "none, partial, heavy (parked cars, other pedestrians)", "all three levels (measured below)"),
], columns=["Attribute", "Specified ODD", "Evidence in §4"])
for k, v in ITEM.items():
    print(f"{k:16s}: {v}")
display(ODD)

## 2. HARA summary and safety goals

The four hazardous events and their ratings are copied unchanged from [03-02](../03_Functional_Safety_ISO26262/02_hara.ipynb), which is the source of truth (rationale is given there). The ASILs are recomputed with `avsafety.iso26262.determine_asil`, and each safety goal takes the highest ASIL of the events it covers.

In [ ]:
hara = pd.DataFrame([
    ("HE-01", "MB-01 omission", "Urban, 30-50 km/h, pedestrian steps into path", "S3", "E2", "C2"),
    ("HE-02", "MB-01 omission", "Rural/motorway, approaching stationary traffic jam end", "S3", "E3", "C2"),
    ("HE-03", "MB-02 unintended braking", "Motorway, 100-130 km/h, follower at ~1 s headway", "S3", "E4", "C3"),
    ("HE-04", "MB-02 unintended braking", "Urban, 30-50 km/h, follower close behind", "S1", "E4", "C2"),
], columns=["HE", "Malfunctioning behaviour", "Operational situation", "S", "E", "C"])
hara["ASIL"] = [I26262.determine_asil(r.S, r.E, r.C) for r in hara.itertuples()]

SG_DEF = {  # statement, safe state, FTTI (ms) -- as in 03-02 (FTTI illustrative)
    "MB-01": ("SG-02", "Avoid omission of AEB deceleration when a collision with a pedestrian or a stationary vehicle is imminent",
              "AEB unavailable, driver warned", 500),
    "MB-02": ("SG-01", "Avoid unintended AEB deceleration", "AEB brake request inhibited, driver warned", 300),
}
rows = []
for mb, grp in hara.groupby(hara["Malfunctioning behaviour"].str[:5]):
    sg, text, safe, ftti = SG_DEF[mb]
    rows.append({"SG": sg, "Safety goal": text, "ASIL": max(grp.ASIL, key=I26262.asil_rank),
                 "Covers": ", ".join(grp.HE), "Safe state": safe, "FTTI (ms)": ftti})
goals = pd.DataFrame(rows).sort_values("SG").reset_index(drop=True)
display(hara)
display(goals)

assert goals.set_index("SG").loc["SG-01", ["ASIL", "FTTI (ms)"]].tolist() == ["D", 300]
assert goals.set_index("SG").loc["SG-02", ["ASIL", "FTTI (ms)"]].tolist() == ["B", 500]
uncovered = [h for h, a in zip(hara.HE, hara.ASIL) if a != "QM" and not goals.Covers.str.contains(h).any()]
for r in goals.itertuples():
    print(f"{r.SG} [ASIL {r.ASIL}, FTTI {r._6} ms] {r._2} (covers {r.Covers})")
print("ASIL-rated hazardous events without a safety goal:", uncovered or "none")

The pedestrian hazard HE-01 alone is only ASIL A; SG-02 is ASIL B because it also covers the stationary-vehicle event HE-02. The **ASIL D** comes from the *opposite* failure, unintended braking. This tension drives everything below: a detector tuned to never miss a pedestrian produces more ghost objects, and ghost objects are the cause of the ASIL D hazard.

## 3. Functional safety concept: decomposition and a fault-tree check

**Functional safety requirements (FSR).** SG-01 is decomposed **D → B(D) + B(D)** onto a decision channel and an independent plausibility channel on the brake ECU, exactly as in [03-02 §10](../03_Functional_Safety_ISO26262/02_hara.ipynb). Timing of the credited safety mechanisms is checked against the FTTI (FDTI + FRTI ≤ FTTI).

**Fault-tree check.** The top event *"unintended AEB deceleration reaches the brakes"* is evaluated with a small fault tree. The DFA in 03-02 found two coupling factors shared by both channels: the **front radar** and the **vehicle-speed signal**. In the tree they appear as single-event cut sets, which is exactly what a decomposition must not have. Method: [03-04 FTA](../03_Functional_Safety_ISO26262/04_fault_tree_analysis.ipynb); hardware metrics: [03-03 FMEDA](../03_Functional_Safety_ISO26262/03_fmea_and_fmeda.ipynb).

> **Illustrative data:** failure rates (FIT = failures per 10⁹ h), diagnostic coverages and timings are teaching values.

In [ ]:
sg01_asil = goals.set_index("SG").loc["SG-01", "ASIL"]
DECOMP = ("B(D)", "B(D)")
assert DECOMP in I26262.decomposition_options(sg01_asil)

fsr = pd.DataFrame([
    ("FSR-01", "SG-01", "Decision channel issues a deceleration request only if the fused object list shows an imminent collision", DECOMP[0], "ADAS ECU"),
    ("FSR-02", "SG-01", "Plausibility channel forwards a request only if raw radar range-rate and ego speed confirm it", DECOMP[1], "Brake ECU"),
    ("FSR-03", "SG-01", "Shared inputs (radar targets, ego speed) are protected end-to-end (E2E) with detection within FDTI", sg01_asil, "Radar, gateway"),
    ("FSR-04", "SG-02", "Loss of camera frames or perception output is detected and AEB-unavailable is indicated", goals.ASIL[1], "ADAS ECU"),
    ("FSR-05", "SG-02", "Pedestrian detection meets performance requirement PR-01 inside the ODD (SOTIF, §4)", goals.ASIL[1] + " / SOTIF", "Perception"),
], columns=["FSR", "SG", "Requirement", "ASIL", "Allocated to"])
display(fsr)

mech = pd.DataFrame([
    ("SG-01", "Brake-ECU plausibility check (FSR-02)", 50, 150),
    ("SG-01", "E2E check on radar and speed signals (FSR-03)", 40, 150),
    ("SG-02", "Camera frame-loss detection, AEB-unavailable warning (FSR-04)", 100, 200),
], columns=["SG", "Mechanism", "FDTI (ms)", "FRTI (ms)"]).merge(goals[["SG", "FTTI (ms)"]], on="SG")
mech["FHTI (ms)"] = mech["FDTI (ms)"] + mech["FRTI (ms)"]
mech["Within FTTI"] = mech["FHTI (ms)"] <= mech["FTTI (ms)"]
display(mech)
TIMING_OK = bool(mech["Within FTTI"].all())
print("All credited mechanisms within FTTI:", TIMING_OK)

In [ ]:
# Mini fault tree for the SG-01 top event. Gates are nested tuples ("AND"|"OR", children...).
FIT = 1e-9                       # 1 FIT = 1e-9 per hour
T_LATENT_H = 1.0                 # latent faults of the plausibility channel are found at the next power-up self-test
BASIC = {                        # rate (FIT), latent?  -- illustrative
    "DEC": (200, False),         # decision channel issues an erroneous request (ADAS ECU hardware)
    "PLAUS": (100, True),        # plausibility channel fails to veto (latent until self-test)
    "RADAR": (30, False),        # shared front radar reports a ghost target with plausible range-rate
    "SPEED": (50, False),        # shared ego-speed signal corrupted
}
TREE = ("OR", ("AND", "DEC", "PLAUS"), "RADAR", "SPEED")

def cut_sets(node):
    if isinstance(node, str):
        return [frozenset([node])]
    gate, *kids = node
    sets = [cut_sets(k) for k in kids]
    if gate == "OR":
        out = [c for s in sets for c in s]
    else:
        out = [frozenset().union(*combo) for combo in itertools.product(*sets)]
    return [c for c in set(out) if not any(o < c for o in out)]          # keep minimal sets

def cut_set_rate(cs, rates):
    """First-order rate per hour: one event is the trigger, the others must already be latent."""
    if len(cs) == 1:
        return rates[next(iter(cs))] * FIT
    total = 0.0
    for trig in cs:
        p_others = np.prod([rates[e] * FIT * T_LATENT_H / 2 for e in cs if e != trig and BASIC[e][1]] or [0.0])
        total += rates[trig] * FIT * p_others if not BASIC[trig][1] else 0.0
    return total

def top_event_rate(dc):
    rates = {e: r * (1 - dc.get(e, 0.0)) for e, (r, _) in BASIC.items()}   # residual (undetected) rate
    mcs = sorted(cut_sets(TREE), key=len)
    return mcs, {", ".join(sorted(c)): cut_set_rate(c, rates) for c in mcs}

target = I26262.PMHF_TARGET_PER_HOUR[sg01_asil]
results = {}
for label, dc in [("without E2E (DFA finding open)", {}), ("with E2E, DC 99 % on RADAR and SPEED (FSR-03)", {"RADAR": 0.99, "SPEED": 0.99})]:
    mcs, per_cs = top_event_rate(dc)
    total = sum(per_cs.values())
    results[label] = total
    print(f"{label}: top event {total:.2e} /h  (target ASIL {sg01_asil}: < {target:.0e} /h) -> {'meets' if total < target else 'VIOLATES'}")
    for cs, r in per_cs.items():
        print(f"    cut set {{{cs}}}: {r:.2e} /h")
singles = [", ".join(c) for c in mcs if len(c) == 1]
FTA_OK = results["with E2E, DC 99 % on RADAR and SPEED (FSR-03)"] < target
print(f"Single-event cut sets: {singles} -> each needs a mechanism developed to ASIL {sg01_asil} (FSR-03)")

The independent channels make the AND branch negligible; the risk sits entirely in the **shared inputs**. Without FSR-03 the decomposition argument fails, with it the check passes. This is a sanity check consistent with the PMHF target, not the PMHF calculation itself, which needs the full FMEDA of 03-03.

## 4. SOTIF: perception performance requirement, measured on KITTI

### 4.1 Deriving the requirement

A pedestrian must be detected early enough for the brake to stop the car. The braking-relevant range is the stopping distance `avsafety.sotif.stopping_distance(v, t_react, a)` with a system reaction time (detection confirmation plus brake build-up) and a deceleration that also holds on a wet road.

**PR-01 (derived from SG-02, SOTIF acceptance criterion).** *For pedestrians within the braking-relevant range, within ±4 m of the vehicle axis, at most partly occluded and at least 25 px tall, the per-frame detection recall of the camera channel shall be ≥ 0.90, demonstrated with 95 % one-sided confidence.*

The recall target is set **before** looking at the data (illustrative policy). It is a per-frame figure; the AEB decision uses several frames, but consecutive frames of the same pedestrian are strongly correlated, so a per-frame requirement must not lean on independence across frames.

In [ ]:
T_REACT_S, A_BRAKE = 0.6, 6.0          # s, m/s^2  (illustrative: 3 frames at 10 Hz + brake build-up; wet asphalt)
R_MIN, CONF = 0.90, 0.95               # PR-01 recall target and confidence (illustrative policy)
LATERAL_M, MIN_HEIGHT_PX, MAX_OCC = 4.0, 25, 1
TAU = 0.5                              # detector operating threshold (design parameter)
SPEEDS_KMH = [30, 40, 50, 60]

rng_tab = pd.DataFrame({"v (km/h)": SPEEDS_KMH})
rng_tab["Braking-relevant range (m)"] = [SOTIF.stopping_distance(v / 3.6, T_REACT_S, A_BRAKE) for v in SPEEDS_KMH]
D_REQ = float(rng_tab["Braking-relevant range (m)"].iloc[-1])
display(rng_tab.round(1))
print(f"PR-01 range at the top of the pedestrian sub-scope (60 km/h): {D_REQ:.1f} m")

### 4.2 Test set: 40 KITTI frames, chosen by rule

Frames are drawn at random (fixed seed) from all KITTI training frames that contain **at least two labelled pedestrians within 40 m**. The rule, not hand-picking, selects them; the list is stored as a constant so the result is reproducible. Odd and even positions form a calibration and a test split for §5.

In [ ]:
FRAMES = [251, 596, 764, 925, 1196, 1252, 1314, 1355, 1358, 1700, 2095, 2124, 2189, 2595, 2609, 2640, 2707, 3359, 3398, 3740,
          4389, 4512, 4569, 4627, 4717, 4740, 4827, 5031, 5115, 5248, 5467, 5571, 5881, 5980, 6070, 6139, 6475, 6627, 6957, 7410]

labels_all = D.kitti_object_labels_all()
ped_all = labels_all[labels_all.type == "Pedestrian"]
n_near = ped_all[ped_all.distance_m <= 40].groupby("frame").size()
candidates = sorted(int(f) for f in n_near[n_near >= 2].index)
assert sorted(np.random.default_rng(SEED).choice(candidates, 40, replace=False).tolist()) == FRAMES

CAL_FRAMES, TEST_FRAMES = FRAMES[0::2], FRAMES[1::2]
gt = labels_all[labels_all.frame.isin([f"{f:06d}" for f in FRAMES])].copy()
gt["frame"] = gt.frame.astype(int)
ped = gt[gt.type == "Pedestrian"].copy()
ped["evaluable"] = ped.box_height_px >= MIN_HEIGHT_PX
print(f"{len(candidates)} candidate frames; {len(FRAMES)} selected; {len(ped)} labelled pedestrians "
      f"({int(ped.evaluable.sum())} at least {MIN_HEIGHT_PX} px tall); occlusion levels: "
      + ", ".join(f"{k}: {v}" for k, v in ped.occluded.value_counts().sort_index().items()))

In [ ]:
import torch
from torchvision.models.detection import FasterRCNN_MobileNet_V3_Large_FPN_Weights, fasterrcnn_mobilenet_v3_large_fpn

torch.manual_seed(SEED)
torch.set_num_threads(min(4, os.cpu_count()))
detector = fasterrcnn_mobilenet_v3_large_fpn(weights=FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT,
                                             box_score_thresh=0.01).eval()
PERSON = 1                                          # COCO label id

images, det_rows, t0 = {}, [], time.time()
try:
    for f in FRAMES:
        img = D.kitti_object(f, parts=("image",)).image
        images[f] = img
        with torch.no_grad():
            out = detector([torch.from_numpy(img.copy()).permute(2, 0, 1).float() / 255])[0]
        keep = out["labels"] == PERSON
        for b, s in zip(out["boxes"][keep].numpy(), out["scores"][keep].numpy()):
            det_rows.append({"frame": f, "x1": b[0], "y1": b[1], "x2": b[2], "y2": b[3], "score": float(s)})
except D.DataUnavailable as err:
    raise RuntimeError(f"KITTI frames could not be downloaded ({err}). Run again with internet access.") from None
det = pd.DataFrame(det_rows)
print(f"{len(FRAMES)} frames, {len(det)} person detections with score >= 0.01, {time.time() - t0:.0f} s")

### 4.3 Matching

Standard KITTI-style rules: a detection matches a labelled pedestrian at **IoU ≥ 0.5**, greedily in order of descending score, one-to-one. Detections that overlap *Person_sitting*, *Cyclist* or *DontCare* regions are ignored rather than counted as false positives. Because the matching is greedy by score, the matching at any threshold τ is a prefix of the matching at 0.01, so each pedestrian gets a single "matched score" *s* and is detected at τ exactly when *s* ≥ τ.

In [ ]:
def iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter)

def overlap_frac(d, region):
    ix = max(0.0, min(d[2], region[2]) - max(d[0], region[0])); iy = max(0.0, min(d[3], region[3]) - max(d[1], region[1]))
    return ix * iy / ((d[2] - d[0]) * (d[3] - d[1]))

ped["score"] = 0.0
det["status"] = "FP"
for f in FRAMES:
    g = ped[ped.frame == f]; gboxes = g[["x1", "y1", "x2", "y2"]].to_numpy(); free = set(g.index)
    others = gt[(gt.frame == f) & gt.type.isin(["Person_sitting", "Cyclist"])][["x1", "y1", "x2", "y2"]].to_numpy()
    dontcare = gt[(gt.frame == f) & (gt.type == "DontCare")][["x1", "y1", "x2", "y2"]].to_numpy()
    for i, d in det[det.frame == f].sort_values("score", ascending=False).iterrows():
        box = d[["x1", "y1", "x2", "y2"]].to_numpy(float)
        best, best_iou = None, 0.5
        for gi, gb in zip(g.index, gboxes):
            v = iou(box, gb)
            if gi in free and v >= best_iou:
                best, best_iou = gi, v
        if best is not None:
            free.discard(best); ped.loc[best, "score"] = d.score
            det.loc[i, "status"] = "TP" if ped.loc[best, "evaluable"] else "ignored"
        elif any(iou(box, o) >= 0.5 for o in others) or any(overlap_frac(box, r) >= 0.5 for r in dontcare):
            det.loc[i, "status"] = "ignored"

ped["detected"] = ped.score >= TAU
ped["in_corridor"] = ped.x.abs() <= LATERAL_M
fp_per_frame = (det[(det.score >= TAU)].status == "FP").sum() / len(FRAMES)
print(f"At tau = {TAU}: recall over all evaluable pedestrians {ped[ped.evaluable].detected.mean():.3f} "
      f"(n = {int(ped.evaluable.sum())}); false positives per frame {fp_per_frame:.2f}")

In [ ]:
def show_frame(ax, f, title):
    ax.imshow(images[f]); ax.set_axis_off(); ax.set_title(title, fontsize=10)
    for r in ped[(ped.frame == f) & ped.evaluable].itertuples():
        c = OI["green"] if r.detected else OI["vermillion"]
        ax.add_patch(Rectangle((r.x1, r.y1), r.x2 - r.x1, r.y2 - r.y1, fill=False, ec=c, lw=2.2))
        ax.text(r.x1, r.y1 - 4, f"{r.distance_m:.0f} m, occ {r.occluded}", color="white", fontsize=7,
                bbox=dict(fc=c, ec="none", pad=1))
    for r in det[(det.frame == f) & (det.score >= TAU) & (det.status == "FP")].itertuples():
        ax.add_patch(Rectangle((r.x1, r.y1), r.x2 - r.x1, r.y2 - r.y1, fill=False, ec=OI["orange"], lw=1.6, ls="--"))

ev = ped[ped.evaluable]
per_frame = ev.groupby("frame").agg(n=("detected", "size"), missed=("detected", lambda s: int((~s).sum())))
f_miss = int(per_frame.sort_values(["missed", "n"], ascending=False).index[0])
f_good = int(per_frame[per_frame.missed == 0].sort_values("n", ascending=False).index[0])
fig, axes = plt.subplots(2, 1, figsize=(13, 8.4))
show_frame(axes[0], f_good, f"KITTI {f_good:06d}: all {per_frame.loc[f_good, 'n']} evaluable pedestrians detected at tau = {TAU}")
show_frame(axes[1], f_miss, f"KITTI {f_miss:06d}: {per_frame.loc[f_miss, 'missed']} of {per_frame.loc[f_miss, 'n']} missed")
handles = [Rectangle((0, 0), 1, 1, fill=False, ec=OI["green"], lw=2), Rectangle((0, 0), 1, 1, fill=False, ec=OI["vermillion"], lw=2),
           Rectangle((0, 0), 1, 1, fill=False, ec=OI["orange"], lw=1.6, ls="--")]
fig.legend(handles, ["labelled pedestrian, detected", "labelled pedestrian, missed", "false positive (person)"],
           loc="lower center", ncol=3, frameon=False)
plt.tight_layout(rect=(0, 0.04, 1, 1)); plt.show()
print(f"Frame {f_good:06d}: no misses. Frame {f_miss:06d}: misses at distances "
      + ", ".join(f"{d:.0f} m (occ {o})" for d, o in ev[(ev.frame == f_miss) & ~ev.detected][["distance_m", "occluded"]].to_numpy()))

### 4.4 Recall by distance and occlusion

The lower bound uses the one-sided Poisson bound on the miss rate, `avsafety.validation.poisson_upper_bound(misses, n)`, which is conservative for a binomial proportion.

In [ ]:
def recall_stats(df):
    n, k = len(df), int(df.detected.sum())
    lo = max(0.0, 1 - VAL.poisson_upper_bound(n - k, n, CONF)) if n else np.nan
    return pd.Series({"n": n, "detected": k, "recall": k / n if n else np.nan, "lower 95 %": lo})

BINS = [0, 10, 20, 30, 40, 80]
ev = ped[ped.evaluable].copy()
ev["distance bin (m)"] = pd.cut(ev.distance_m, BINS, right=False).astype(str)
by_dist = ev[ev.occluded <= MAX_OCC].groupby("distance bin (m)", sort=False).apply(recall_stats, include_groups=False)
by_dist = by_dist.reindex([str(pd.Interval(a, b, closed="left")) for a, b in zip(BINS[:-1], BINS[1:])]).dropna(how="all")
by_occ = ev[ev.distance_m <= D_REQ].groupby("occluded").apply(recall_stats, include_groups=False)
by_occ.index = [{0: "0 fully visible", 1: "1 partly", 2: "2 largely", 3: "3 unknown"}[i] for i in by_occ.index]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.3), sharey=True)
for ax, tab, xl, col in [(axes[0], by_dist, f"Distance (m); occlusion <= {MAX_OCC}", OI["blue"]),
                         (axes[1], by_occ, f"KITTI occlusion level; distance <= {D_REQ:.0f} m", OI["purple"])]:
    x = np.arange(len(tab))
    ax.bar(x, tab.recall, color=col, edgecolor="black", label="recall (point estimate)")
    ax.errorbar(x, tab.recall, yerr=[tab.recall - tab["lower 95 %"], np.zeros(len(tab))], fmt="none", ecolor="black",
                capsize=4, label="95 % lower bound")
    for xi, (r, n) in enumerate(zip(tab.recall, tab.n)):
        ax.text(xi, 0.02, f"n={int(n)}", ha="center", color="white", fontsize=9, fontweight="bold")
    ax.axhline(R_MIN, color=OI["vermillion"], ls="--", label=f"PR-01 target {R_MIN}")
    ax.set_xticks(x); ax.set_xticklabels(tab.index, rotation=0); ax.set_xlabel(xl); ax.set_ylim(0, 1.05)
axes[0].set_ylabel(f"Per-frame recall at tau = {TAU}")
axes[0].legend(loc="lower left", bbox_to_anchor=(0, 0.08), fontsize=8)
fig.suptitle("Pedestrian recall of a COCO-pretrained detector on 40 KITTI frames", fontweight="bold")
plt.tight_layout(); plt.show()
display(by_dist.round(3)); display(by_occ.round(3))

### 4.5 Requirement check and ODD restriction

PR-01 is evaluated for each candidate speed limit of the pedestrian sub-scope, because a lower speed shortens the braking-relevant range. If PR-01 fails at 60 km/h but holds at a lower speed, the SOTIF measure *restrict the ODD* is available: pedestrian AEB is specified, and claimed, only up to that speed until better evidence exists.

In [ ]:
def pr01_scope(df, d_max):
    return df[df.evaluable & df.in_corridor & (df.occluded <= MAX_OCC) & (df.distance_m <= d_max)]

checks = []
for v, d in zip(rng_tab["v (km/h)"], rng_tab["Braking-relevant range (m)"]):
    s = recall_stats(pr01_scope(ped, d))
    checks.append({"v max (km/h)": v, "range (m)": round(d, 1), **s.to_dict(), "PR-01 met": bool(s["lower 95 %"] >= R_MIN)})
pr01 = pd.DataFrame(checks)
pr01[["n", "detected"]] = pr01[["n", "detected"]].astype(int)
display(pr01.round(3))

PR01_FULL = bool(pr01.loc[pr01["v max (km/h)"] == 60, "PR-01 met"].item())
passing = pr01[pr01["PR-01 met"]]
V_STAR = int(passing["v max (km/h)"].max()) if len(passing) else None
full = pr01.iloc[-1]
print(f"PR-01 at 60 km/h (range {full['range (m)']} m): recall {full.recall:.3f}, lower bound {full['lower 95 %']:.3f} "
      f"on n = {full.n} -> {'met' if PR01_FULL else 'NOT met'}")
print("Highest speed limit at which PR-01 is demonstrated:", f"{V_STAR} km/h" if V_STAR else "none")

In [ ]:
# Performance insufficiencies -> triggering conditions for the validation plan
insuff = []
for name, tab in [("distance", by_dist), ("occlusion", by_occ)]:
    for idx, r in tab.iterrows():
        if r.n and r["lower 95 %"] < R_MIN:
            insuff.append({"Factor": name, "Condition": idx, "n": int(r.n), "recall": round(r.recall, 3),
                           "lower 95 %": round(r["lower 95 %"], 3)})
small = ped[(~ped.evaluable) & (ped.distance_m <= D_REQ)]
insuff = pd.DataFrame(insuff)
display(insuff)
print(f"{len(insuff)} conditions below the PR-01 target. Additionally {len(small)} pedestrian(s) within {D_REQ:.0f} m are "
      f"< {MIN_HEIGHT_PX} px tall (children or truncated) and are outside the evaluable set.")
print("Not testable with this data at all: night, dusk, rain (KITTI is daytime and dry) -> ODD restriction or new evidence.")

**Reading the result.** The measured insufficiencies become entries in the SOTIF triggering-condition analysis ([04-02](../04_SOTIF_ISO21448/02_triggering_conditions_and_scenarios.ipynb)): each needs either a design measure (a better detector, radar-only fallback for confirmed objects, tracking), an ODD restriction, or an argument that the residual risk is acceptable. Heavy occlusion cannot be removed by restricting the ODD, because pedestrians stepping out between parked cars are part of urban traffic; it is a *known hazardous scenario* (ISO 21448 Area 2) for which only the design can help. Night and rain are not insufficiencies of this detector; they are **gaps in the evidence**.

## 5. Uncertainty evidence (ISO/PAS 8800)

Two questions, each on the calibration frames (20) and checked on the held-out test frames (20):

1. **Calibration.** If threat assessment or fusion uses the detector score as a probability, the score must be calibrated. Expected calibration error (ECE) before and after temperature scaling, `avsafety.uq` ([06-04](../06_AI_Safety_ISO_PAS_8800/04_calibration.ipynb)).
2. **Conformal recall threshold.** Which score threshold makes the probability of missing a braking-relevant pedestrian at most α = 0.10, distribution-free? Split conformal prediction ([06-05](../06_AI_Safety_ISO_PAS_8800/05_conformal_prediction.ipynb)) with the score *s* of §4.3 and the non-conformity 1 − *s*.

In [ ]:
def detection_set(frames):
    d = det[det.frame.isin(frames) & (det.score >= 0.05) & det.status.isin(["TP", "FP"])]
    s = d.score.to_numpy().clip(1e-6, 1 - 1e-6)
    return np.column_stack([np.zeros_like(s), np.log(s / (1 - s))]), (d.status == "TP").to_numpy().astype(int)

cal_logits, cal_y = detection_set(CAL_FRAMES)
test_logits, test_y = detection_set(TEST_FRAMES)
T_FIT = UQ.fit_temperature(cal_logits, cal_y)
ece_before = UQ.expected_calibration_error(UQ.softmax(test_logits), test_y, n_bins=10)
ece_after = UQ.expected_calibration_error(UQ.softmax(test_logits, T_FIT), test_y, n_bins=10)
ECE_MAX = 0.05                                              # illustrative acceptance

fig, ax = plt.subplots(figsize=(6.2, 4.8))
for probs, lab, col, mk in [(UQ.softmax(test_logits), f"raw score (ECE {ece_before:.3f})", OI["vermillion"], "o"),
                            (UQ.softmax(test_logits, T_FIT), f"temperature T = {T_FIT:.2f} (ECE {ece_after:.3f})", OI["blue"], "s")]:
    b = [(c, a, n) for c, a, n in UQ.reliability_bins(probs, test_y, 10) if n >= 5]
    ax.plot([c for c, _, _ in b], [a for _, a, _ in b], marker=mk, color=col, label=lab)
ax.plot([0.5, 1], [0.5, 1], ls=":", color="black", label="perfect calibration")
ax.set_xlabel("Top-label confidence (bins with >= 5 detections)"); ax.set_ylabel("Observed accuracy")
ax.set_title("Reliability of person scores on test frames", fontweight="bold"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f"Detections (score >= 0.05): {len(cal_y)} calibration, {len(test_y)} test; precision on test {test_y.mean():.2f}")
print(f"ECE on test frames: {ece_before:.3f} raw -> {ece_after:.3f} after temperature scaling (target <= {ECE_MAX})")

In [ ]:
ALPHA = 0.10
def in_scope_scores(frames):
    return pr01_scope(ped[ped.frame.isin(frames)], D_REQ).score.to_numpy()

s_cal, s_test = in_scope_scores(CAL_FRAMES), in_scope_scores(TEST_FRAMES)
q = UQ.conformal_threshold(np.column_stack([1 - s_cal, s_cal]), np.ones(len(s_cal), int), alpha=ALPHA)
TAU_CONF = 1 - q
covered = UQ.conformal_sets(np.column_stack([1 - s_test, s_test]), q)[:, 1]
cov_test = covered.mean()
cov_tol = 2 * np.sqrt(ALPHA * (1 - ALPHA) / len(s_test))
CONFORMAL_OK = bool(TAU_CONF > 0.01 and cov_test >= 1 - ALPHA - cov_tol)
fp_conf = (det[det.frame.isin(TEST_FRAMES) & (det.score >= max(TAU_CONF, 0.01))].status == "FP").sum() / len(TEST_FRAMES)
fp_tau = (det[det.frame.isin(TEST_FRAMES) & (det.score >= TAU)].status == "FP").sum() / len(TEST_FRAMES)
print(f"Calibration pedestrians in PR-01 scope: {len(s_cal)}, of which {int((s_cal == 0).sum())} have no detection at any score")
print(f"Conformal threshold for alpha = {ALPHA}: tau_conf = {TAU_CONF:.3f}")
print(f"Held-out recall at tau_conf: {cov_test:.3f} on {len(s_test)} pedestrians (target {1 - ALPHA:.2f}, sampling tolerance {cov_tol:.2f})")
print(f"False positives per test frame: {fp_tau:.2f} at tau = {TAU} vs {fp_conf:.2f} at tau_conf")

**Reading the result.** Temperature scaling changes the scores, never their ranking, so it does not change recall; it only makes the scores usable as probabilities downstream. The conformal threshold gives a recall statement with a finite-sample guarantee, **but** only under exchangeability: here, pedestrians from the same 20 daytime Karlsruhe frames. It says nothing about night, rain or another city, and pedestrians in one frame are not independent. The price of the lower threshold is the rise in false positives: those feed the ASIL D hazard (SG-01) and must be stopped by the plausibility channel of §3. Detector threshold, recall and decomposition are therefore one design decision, not three.

## 6. Cybersecurity: TARA summary for the perception → brake path

The four threat scenarios of [05-02](../05_Cybersecurity_ISO21434/02_tara.ipynb) that act on this path are recomputed with `avsafety.iso21434`. Factor values are those of the most feasible attack path in 05-02 (step-level breakdown there); controls raise individual factors to a "floor" or change the damage scenario. The CAL uses impact × the most remote attack vector of all paths.

> **Illustrative data:** attack-potential factors and the acceptance policy (residual risk ≤ 2) are the teaching values of 05-02.

In [ ]:
TABLES = {"elapsed_time": I21434.ELAPSED_TIME, "expertise": I21434.EXPERTISE, "knowledge": I21434.KNOWLEDGE,
          "window": I21434.WINDOW, "equipment": I21434.EQUIPMENT}
def raise_to(factors, floors):
    return {k: (floors[k] if k in floors and TABLES[k][floors[k]] > TABLES[k][v] else v) for k, v in factors.items()}

THREATS = [
    ("TS-01", "Forged brake-request frames on CAN-FD", "SG-01", "Severe",
     dict(elapsed_time="<=1 week", expertise="proficient", knowledge="restricted", window="moderate", equipment="specialized"),
     "Network", "SecOC with keys in HSM; brake-ECU plausibility", dict(knowledge="strictly confidential", expertise="expert"), "Severe"),
    ("TS-02", "Manipulated DNN weights via a malicious update", "SG-02", "Severe",
     dict(elapsed_time="<=6 months", expertise="expert", knowledge="confidential", window="moderate", equipment="bespoke"),
     "Network", "Signed updates, secure boot (UN R156 / ISO 24089)", {}, "Severe"),
    ("TS-03", "Physical adversarial patch suppresses pedestrian detections", "SG-02", "Severe",
     dict(elapsed_time="<=1 month", expertise="expert", knowledge="restricted", window="easy", equipment="standard"),
     "Physical", "Independent radar confirmation; perception plausibility monitor",
     dict(elapsed_time="<=6 months", expertise="multiple experts", equipment="bespoke"), "Severe"),
    ("TS-04", "Camera dazzled by a high-power light source", "SG-02", "Severe",
     dict(elapsed_time="<=1 day", expertise="proficient", knowledge="public", window="moderate", equipment="standard"),
     "Physical", "Blinding detection -> degraded mode and driver warning", {}, "Major"),
]
ACCEPTABLE_RESIDUAL = 2
RISK_ACCEPTANCE_RECORDS = {}          # threat scenario -> signed acceptance record; none exists yet

rows = []
for ts, name, sg, impact, f, remote, control, floors, impact_after in THREATS:
    ap = I21434.attack_potential(**f); feas = I21434.feasibility_from_potential(ap)
    ap2 = I21434.attack_potential(**raise_to(f, floors)); feas2 = I21434.feasibility_from_potential(ap2)
    residual = I21434.risk_value(impact_after, feas2)
    rows.append({"TS": ts, "Threat": name, "SG": sg, "AP": ap, "Feasibility": feas, "Risk": I21434.risk_value(impact, feas),
                 "CAL": I21434.determine_cal(impact, remote), "Control": control, "AP after": ap2, "Residual": residual,
                 "Accepted": residual <= ACCEPTABLE_RESIDUAL or ts in RISK_ACCEPTANCE_RECORDS})
tara = pd.DataFrame(rows)
assert tara.Risk.tolist() == [4, 2, 4, 5] and tara.CAL.tolist() == [4, 4, 2, 2] and tara.Residual.tolist() == [2, 2, 2, 4]  # = 05-02
display(tara)
CYBER_OK = bool(tara.Accepted.all())
for r in tara.itertuples():
    print(f"{r.TS} ({r.SG}): risk {r.Risk} -> residual {r.Residual}, CAL {r.CAL}; {'acceptable' if r.Accepted else 'OPEN: needs treatment or a signed risk acceptance'}")

TS-03 and TS-04 attack the same weakness that §4 measured: the camera's ability to see a pedestrian. Dazzling is the deliberate version of the SOTIF triggering condition *low sun glare*, and the adversarial patch is the deliberate version of *unusual appearance*. Their residual risk depends on the radar confirmation and degradation concept, which is why the TARA and the SOTIF analysis must share one item definition ([08-01](01_standards_integration.ipynb)).

## 7. Validation target and scenario coverage

ISO 21448 asks for **validation targets** derived from the acceptance criteria. Two are needed for pedestrian AEB:

* **V-01 (SG-01, false activation):** unintended AEB deceleration > 3 m/s² at most once per 10⁶ km of driving. Demonstrated by failure-free (or few-failure) fleet exposure, `avsafety.validation.miles_to_demonstrate` (the formula is unit-free; here km).
* **V-02 (SG-02, omission):** probability of missing a braking-relevant pedestrian encounter in the ODD at most 10⁻². The "exposure" is now the number of encounters, not kilometres.

> **Illustrative data:** both targets and the fleet size are teaching values; real targets come from the acceptance criteria (e.g. GAMAB / positive risk balance arguments, field data of comparable systems).

In [ ]:
V01_RATE_PER_KM, V02_MISS_PROB = 1e-6, 1e-2
FLEET, KM_PER_VEHICLE_YEAR = 20, 100_000

val = []
for k in (0, 1, 2):
    km = VAL.miles_to_demonstrate(V01_RATE_PER_KM, CONF, failures=k)
    enc = VAL.miles_to_demonstrate(V02_MISS_PROB, CONF, failures=k)
    val.append({"failures allowed": k, "V-01: km needed": f"{km:,.0f}",
                "V-01: years with fleet": round(VAL.fleet_years(km, FLEET, KM_PER_VEHICLE_YEAR), 2),
                "V-02: encounters needed": int(np.ceil(enc))})
val = pd.DataFrame(val)
display(val)

kitti_scope = pr01_scope(ped, D_REQ)
kitti_ub = VAL.poisson_upper_bound(int((~kitti_scope.detected).sum()), len(kitti_scope), CONF)
FIELD_KM_ACHIEVED, ENCOUNTERS_ACHIEVED = 0, 0              # no vehicle-level validation has been run in this notebook
VALIDATION_OK = (FIELD_KM_ACHIEVED >= VAL.miles_to_demonstrate(V01_RATE_PER_KM, CONF)
                 and ENCOUNTERS_ACHIEVED >= VAL.miles_to_demonstrate(V02_MISS_PROB, CONF))
print(f"KITTI evidence: {len(kitti_scope)} pedestrian instances (frames, not encounters); per-frame miss rate upper bound "
      f"{kitti_ub:.3f} vs V-02 target {V02_MISS_PROB} -> component evidence only, not vehicle-level validation")
print(f"Pedestrian conflicts are rated E2 (a few per year per driver): {val.iloc[0]['V-02: encounters needed']} real encounters "
      "cannot be collected on public roads -> proving ground and simulation are required")

Real encounters are too rare, so V-02 is demonstrated with **scenario-based testing** (proving ground for anchor points, simulation for breadth; [04-04](../04_SOTIF_ISO21448/04_simulation_based_validation.ipynb)). Its adequacy is argued by **coverage** of the ODD parameters (ISO 34502). Below, pairwise coverage (`avsafety.coverage.nwise_coverage`) of the evidence in §4 is compared with a planned simulation set built greedily to cover every pair of parameter values.

In [ ]:
DOMAINS = {"ego_speed": ["<=30", "31-45", "46-60"], "pedestrian": ["adult", "child"], "occlusion": ["none", "partial", "heavy"],
           "lighting": ["day", "dusk", "night"], "weather": ["dry", "rain"], "distance": ["<15 m", "15-30 m", ">=30 m"]}
OCC_NAME = {0: "none", 1: "partial", 2: "heavy", 3: "heavy"}
kitti_scen = [{"pedestrian": "child" if r.h < 1.40 else "adult", "occlusion": OCC_NAME[r.occluded], "lighting": "day",
               "weather": "dry", "distance": "<15 m" if r.distance_m < 15 else ("15-30 m" if r.distance_m < 30 else ">=30 m")}
              for r in ped[ped.evaluable].itertuples()]       # ego speed is not observable in single frames
cov_kitti = COV.nwise_coverage(DOMAINS, kitti_scen, n=2)

def greedy_pairwise(domains, seed, n_candidates=60):
    rs = np.random.default_rng(seed)
    params = sorted(domains)
    todo = {((a, va), (b, vb)) for a, b in itertools.combinations(params, 2) for va in domains[a] for vb in domains[b]}
    plan = []
    while todo:
        cands = [{p: domains[p][rs.integers(len(domains[p]))] for p in params} for _ in range(n_candidates)]
        gain = lambda s: sum(((a, s[a]), (b, s[b])) in todo for a, b in itertools.combinations(params, 2))
        best = max(cands, key=gain)
        if gain(best) == 0:                                   # force one uncovered pair into a candidate
            (a, va), (b, vb) = sorted(todo)[0]; best = dict(best, **{a: va, b: vb})
        plan.append(best)
        todo -= {((a, best[a]), (b, best[b])) for a, b in itertools.combinations(params, 2)}
    return plan

sim_plan = greedy_pairwise(DOMAINS, SEED)
cov_sim = COV.nwise_coverage(DOMAINS, sim_plan, n=2)
COVERAGE_EXECUTED = cov_kitti["coverage"]                     # only the KITTI evidence has actually been executed
print(f"Full factorial: {COV.full_factorial_size(DOMAINS)} concrete scenarios")
print(f"KITTI evidence (§4): pairwise coverage {cov_kitti['coverage']:.1%} ({cov_kitti['covered']}/{cov_kitti['total']} pairs)")
print(f"Planned simulation set: {len(sim_plan)} scenarios, pairwise coverage {cov_sim['coverage']:.0%}")
missing_params = pd.Series([p for pair in cov_kitti["missing"] for p, _ in pair]).value_counts()
print("Uncovered pairs in the KITTI evidence involve:", ", ".join(f"{k} ({v})" for k, v in missing_params.items()))
display(pd.DataFrame(sim_plan).head(8))

## 8. Safety case summary in GSN terms

The argument has the structure of [08-04](04_safety_case_gsn.ipynb): a top goal argued over the four sources of hazardous behaviour, plus validation. The table is generated from the results above. **Status** follows GSN practice: *supported* (evidence positive), *challenged* (evidence exists and is negative: a rebutting defeater) or *undeveloped* (evidence still missing).

In [ ]:
NB = {"hara": "../03_Functional_Safety_ISO26262/02_hara.ipynb", "fmeda": "../03_Functional_Safety_ISO26262/03_fmea_and_fmeda.ipynb",
      "fta": "../03_Functional_Safety_ISO26262/04_fault_tree_analysis.ipynb", "vv": "../03_Functional_Safety_ISO26262/06_verification_and_validation.ipynb",
      "odd": "../01_Perception/01_sae_levels_and_odd.ipynb", "det": "../01_Perception/04_object_detection.ipynb",
      "vru": "../01_Perception/05_vulnerable_road_user_detection.ipynb", "tc": "../04_SOTIF_ISO21448/02_triggering_conditions_and_scenarios.ipynb",
      "sim": "../04_SOTIF_ISO21448/04_simulation_based_validation.ipynb", "cal": "../06_AI_Safety_ISO_PAS_8800/04_calibration.ipynb",
      "cp": "../06_AI_Safety_ISO_PAS_8800/05_conformal_prediction.ipynb", "stat": "../06_AI_Safety_ISO_PAS_8800/08_statistical_safety_validation.ipynb",
      "tara": "../05_Cybersecurity_ISO21434/02_tara.ipynb", "ids": "../05_Cybersecurity_ISO21434/04_can_bus_intrusion_detection.ipynb",
      "adv": "../02_Failures_and_Robustness/04_adversarial_attacks.ipynb", "rm": "../07_Runtime_Assurance_and_Operations/01_odd_and_runtime_monitoring.ipynb",
      "int": "01_standards_integration.ipynb", "gsn": "04_safety_case_gsn.ipynb"}
link = lambda *keys: ", ".join(f"[{NB[k].split('/')[-1][:-6]}]({NB[k]})" for k in keys)
status = lambda ok, missing=False: "supported" if ok else ("undeveloped" if missing else "**challenged**")

case = pd.DataFrame([
    ("C1", "Context", "Item, ODD (pedestrian sub-scope <= 60 km/h) and HARA", "§1, §2: safety goals SG-01 (D), SG-02 (B)", link("hara", "odd", "int"), "context"),
    ("G2.1", "Goal", "SG-01 is met: E/E faults do not cause unintended braking", f"§3: B(D)+B(D), FTA {results['with E2E, DC 99 % on RADAR and SPEED (FSR-03)']:.1e}/h, FHTI <= FTTI",
     link("fta", "fmeda", "vv"), status(FTA_OK and TIMING_OK)),
    ("G2.2", "Goal", "SG-02 E/E part: loss of perception is detected within FTTI", "§3: frame-loss mechanism timing", link("hara", "vv"), status(TIMING_OK)),
    ("G3.1", "Goal", "PR-01 pedestrian recall holds in the specified ODD (<= 60 km/h)", f"§4: KITTI, n = {int(full.n)}, lower bound {full['lower 95 %']:.3f}",
     link("vru", "det", "tc"), status(PR01_FULL)),
    ("G3.2", "Goal", f"PR-01 holds in the restricted ODD (<= {V_STAR} km/h, daytime, dry)" if V_STAR else "PR-01 holds in a restricted ODD",
     "§4.5 speed-dependent check", link("vru", "tc", "rm"), status(V_STAR is not None)),
    ("G4.1", "Goal", "Detector scores are calibrated where used as probabilities", f"§5: ECE {ece_after:.3f} after temperature scaling", link("cal"), status(ece_after <= ECE_MAX)),
    ("G4.2", "Goal", "A score threshold bounds the miss rate (exchangeable data)", f"§5: tau_conf {TAU_CONF:.2f}, held-out recall {cov_test:.2f}", link("cp"), status(CONFORMAL_OK)),
    ("G5", "Goal", "Cyber threats to the perception -> brake path are treated", f"§6: residual risks {tara.Residual.tolist()}", link("tara", "ids", "adv"), status(CYBER_OK)),
    ("G6.1", "Goal", "Validation targets V-01/V-02 are met", "§7: no fleet km, no encounters yet", link("stat", "sim"), status(VALIDATION_OK, missing=True)),
    ("G6.2", "Goal", "Executed tests cover the ODD pairwise", f"§7: executed coverage {COVERAGE_EXECUTED:.0%}", link("sim", "tc"), status(COVERAGE_EXECUTED >= 1.0, missing=True)),
], columns=["Node", "Type", "Claim", "Evidence produced here", "Method / course notebook", "Status"])
md_table = "\n".join(["| " + " | ".join(case.columns) + " |", "|" + "---|" * len(case.columns)]
                     + ["| " + " | ".join(str(v) for v in row) + " |" for row in case.itertuples(index=False)])
display(Markdown(md_table))
print("Argument structure and checks: see", NB["gsn"])
print("Status count:", case.Status.str.replace("*", "", regex=False).value_counts().to_dict())

## 9. Release decision

The release criteria are fixed in advance in the safety plan, together with a **decision rule**:

* **blocking** criteria must pass for any release;
* **ODD-restrictable** criteria may fail for the full ODD if they pass in a restricted ODD that is enforced by design (here: pedestrian AEB speed limit, ODD monitor) and communicated to the driver;
* **advisory** criteria become open items (defeaters) in the safety case.

Result: *release* if all pass; *release with ODD restriction* if only restrictable criteria fail and the restricted ODD passes; otherwise *do not release*.

In [ ]:
chk = pd.DataFrame([
    ("R1", "Every ASIL-rated hazardous event has a safety goal", "blocking", not uncovered, "evidence", "§2"),
    ("R2", "Decomposition valid and FTA top event below ASIL D target", "blocking", FTA_OK, "evidence", "§3"),
    ("R3", "All credited safety mechanisms within FTTI", "blocking", TIMING_OK, "evidence", "§3"),
    ("R4", "PR-01 met in the specified ODD (<= 60 km/h)", "ODD-restrictable", PR01_FULL, "evidence", "§4.5"),
    ("R5", "Score calibration ECE <= 0.05 on held-out frames", "advisory", ece_after <= ECE_MAX, "evidence", "§5"),
    ("R6", "Conformal recall threshold verified on held-out frames", "advisory", CONFORMAL_OK, "evidence", "§5"),
    ("R7", "All cybersecurity residual risks acceptable or formally accepted", "blocking", CYBER_OK, "evidence", "§6"),
    ("R8", "Validation targets V-01 and V-02 demonstrated", "blocking", VALIDATION_OK, "missing", "§7"),
    ("R9", "Executed scenarios cover the ODD pairwise (100 %)", "blocking", COVERAGE_EXECUTED >= 1.0, "missing", "§7"),
], columns=["ID", "Criterion", "Category", "Pass", "Basis", "Section"])
chk["Result"] = np.where(chk.Pass, "PASS", "FAIL")
chk["Gap"] = np.where(chk.Pass, "-", np.where(chk.Basis == "missing", "evidence not yet produced", "evidence is negative"))

def decide(c, restricted_ok):
    fail = c[~c.Pass]
    if (fail.Category == "blocking").any():
        return "DO NOT RELEASE"
    if (fail.Category == "ODD-restrictable").any():
        return "RELEASE WITH ODD RESTRICTION" if restricted_ok else "DO NOT RELEASE"
    return "RELEASE"

DECISION = decide(chk, V_STAR is not None)
display(chk[["ID", "Criterion", "Category", "Result", "Gap", "Section"]])
print(f"DECISION: {DECISION}")
for r in chk[~chk.Pass].itertuples():
    print(f"  {r.ID} {r.Result} [{r.Category}] {r.Criterion}: {r.Gap}")

hypo = chk.copy(); hypo.loc[hypo.Gap == "evidence not yet produced", "Pass"] = True
restriction = f"pedestrian AEB claimed up to {V_STAR} km/h, daytime, dry" if V_STAR else "no restriction found"
print(f"\nIf the missing evidence (R8, R9) were produced with positive results, the same rule would give: "
      f"{decide(hypo, V_STAR is not None)} ({restriction}); remaining open items: "
      + (", ".join(hypo[~hypo.Pass].ID) or "none"))

### Why this evidence would not suffice in reality

The decision above is computed honestly from what this notebook contains, and that is far less than a release needs:

* **Sample size and representativeness.** 40 daytime frames from one city cannot establish a recall bound for a product sold in many countries. A real campaign uses tens of thousands of labelled pedestrian instances, stratified by the ODD attributes (night, rain, children, clothing, posture, occlusion type), with labels audited for quality.
* **Frames are not encounters.** PR-01 is a component requirement; SG-02 is a vehicle-level goal. The link between them (tracking, fusion with radar, the decision logic) is verified by closed-loop tests on a proving ground (e.g. Euro NCAP-style VRU scenarios) and in validated simulation.
* **The product detector.** A COCO-pretrained research model is a stand-in. The released model, its version and its training data are what the evidence must describe; any retraining invalidates §4–§5 ([08-04](04_safety_case_gsn.ipynb), stale evidence).
* **Hardware and software evidence.** The FTA check replaces neither the FMEDA nor the PMHF calculation, the DFA, nor requirements-based and fault-injection testing on target hardware.
* **Independent confirmation.** ISO 26262-2 requires confirmation reviews and a functional safety assessment; UL 4600 expects an independent assessment of the safety case. A notebook can prepare these; it cannot replace them.
* **Field monitoring.** After release, safety performance indicators (false activations per 10⁶ km, ODD exits) keep the argument valid ([07-01](../07_Runtime_Assurance_and_Operations/01_odd_and_runtime_monitoring.ipynb)).

## In industry

**Work product.** The release recommendation for a start of production (SOP) or a software update: the safety case summary with its evidence register, the functional safety assessment report (ISO 26262-2), the SOTIF achievement evaluation (ISO 21448 cl. 12), the cybersecurity case (ISO/SAE 21434 cl. 6) and, for UNECE markets, the documentation for the CSMS/SUMS-based type approval. The checklist of §9 corresponds to the release gate in the product development process.

**Who produces it.** The functional safety manager and the SOTIF lead own the argument; the perception team produces the KPI evidence (§4–§5); the cybersecurity manager supplies the TARA and cybersecurity case; validation engineering owns §7. An independent assessor (organisationally independent for ASIL D) reviews the case, and a release board signs the decision, including any explicit residual-risk acceptance.

**What reviewers and assessors look for**

- [ ] Traceability without gaps: hazardous event → safety goal → FSR → evidence, and damage scenario → threat → goal → control.
- [ ] Perception requirements derived from vehicle dynamics and the ODD, with the target and statistical method fixed **before** the test.
- [ ] Test data that represent the ODD (and the evidence of that), with the model version under test equal to the version released.
- [ ] ODD restrictions enforced by design (speed gating, ODD monitor) and communicated to the driver, not only written down.
- [ ] Independence of decomposed channels shown by a DFA, and no single-point cut sets left uncovered.
- [ ] Validation targets with a stated derivation, the exposure achieved, and the coverage argument for simulation including its validity.
- [ ] Every open item visible as an undeveloped goal or defeater, with an owner and a date.

## Key takeaways

1. A release decision is the end of a traceable chain: item and ODD → HARA → safety goals → safety concept → verified requirements → validation → safety case. Breaks in the chain, not individual numbers, usually stop a release.
2. For pedestrian AEB the **ASIL D** goal is about *unintended* braking, while the pedestrian requirement concerns *omission*. Detector threshold, recall and the decomposed plausibility channel are therefore one design decision.
3. Perception requirements can be derived from physics (stopping distance) and checked on real data. Here, a COCO detector on KITTI shows clear dependence on distance and occlusion; the computed result decides whether a speed restriction of the ODD helps.
4. Calibration and conformal prediction are useful evidence with narrow scope: calibration does not change recall, and the conformal guarantee holds only for data exchangeable with the calibration set.
5. Cybersecurity threats on the camera (dazzling, patches) are the deliberate twins of SOTIF triggering conditions and belong in the same argument.
6. Statistical validation targets make clear why fleet driving alone cannot validate pedestrian AEB, and why coverage-driven scenario testing is needed.
7. An honest computed checklist separates *negative evidence* from *missing evidence*. Both stop a release; only the first says something about the design.

## Self-assessment

1. Why is the pedestrian omission hazard HE-01 only ASIL A, while SG-02 is ASIL B?
2. In §3, which cut sets dominate the top event, and which FSR removes them?
3. Recompute the braking-relevant range for 50 km/h with a reaction time of 1.0 s and 8 m/s². How does a longer range affect PR-01?
4. Why must the PR-01 target and the confidence level be fixed before the KITTI evaluation?
5. Temperature scaling reduced (or did not reduce) the ECE in §5. Why does it never change the recall at a fixed threshold, and when is calibration still required?
6. Under which conditions does the conformal threshold of §5 guarantee recall ≥ 0.9, and which two of them are violated when the car drives at night in another city?
7. Why can TS-04 (dazzling) not be reduced to an acceptable residual risk by raising attack potential?
8. Explain the difference between "evidence is negative" and "evidence not yet produced" for the release decision.

<details>
<summary>Answers</summary>

1. HE-01 is S3/E2/C2 → A: a pedestrian conflict is rare (E2) and the driver remains responsible. SG-02 also covers HE-02 (stationary traffic jam end, S3/E3/C2 → B) and takes the highest ASIL of the events it covers.
2. The single-event cut sets {RADAR} and {SPEED}: the shared inputs bypass the decomposition. The AND of the two channels is negligible. FSR-03 (end-to-end protection with high diagnostic coverage, developed to ASIL D) removes them.
3. `stopping_distance(50/3.6, 1.0, 8.0)` = 13.9 + 12.1 ≈ 26.0 m (vs 24.4 m with 0.6 s and 6 m/s²). A longer range includes more distant, smaller pedestrians, for which recall is lower, so PR-01 becomes harder to meet.
4. Otherwise the target can be (consciously or not) tuned to the result, and the test no longer tests anything. Assessors check that acceptance criteria predate the evidence.
5. Temperature scaling is a monotonic transformation of the score; the set of detections above a correspondingly transformed threshold is the same, so ranking-based quantities (recall, precision at matched thresholds) are unchanged. Calibration is still required whenever a downstream component treats the score as a probability, e.g. fusion or a risk-based threat assessment.
6. The calibration and test pedestrians must be exchangeable with the deployment data, and the scores must come from the same model. At night in another city the data distribution changes (exchangeability is violated), and pedestrians within a frame or sequence are dependent; the guarantee no longer applies.
7. The attack needs only a strong light and no knowledge of the system: no vehicle control can raise those factors. The control instead changes the damage (degraded mode, driver warned), which lowers the impact from Severe to Major but leaves residual risk 4. It needs either a different design (e.g. radar-only braking for confirmed objects) or a documented, signed risk acceptance.
8. Negative evidence (e.g. PR-01 not met at 60 km/h) says the design does not meet a requirement and must change, or the ODD must be restricted. Missing evidence (e.g. no validation kilometres) says nothing about the design; the claim is simply unsupported. Both prevent release; they lead to different actions.
</details>

## References

1. Geiger, A., Lenz, P., Urtasun, R. (2012). Are we ready for autonomous driving? The KITTI vision benchmark suite. *IEEE CVPR*, 3354–3361. https://doi.org/10.1109/CVPR.2012.6248074
2. Geiger, A., Lenz, P., Stiller, C., Urtasun, R. (2013). Vision meets robotics: The KITTI dataset. *International Journal of Robotics Research*, 32(11), 1231–1237. https://doi.org/10.1177/0278364913491297
3. Ren, S., He, K., Girshick, R., Sun, J. (2017). Faster R-CNN: Towards real-time object detection with region proposal networks. *IEEE TPAMI*, 39(6), 1137–1149. https://doi.org/10.1109/TPAMI.2016.2577031
4. Howard, A. et al. (2019). Searching for MobileNetV3. *IEEE/CVF ICCV*, 1314–1324. https://doi.org/10.1109/ICCV.2019.00140
5. Lin, T.-Y. et al. (2014). Microsoft COCO: Common objects in context. *ECCV 2014*, LNCS 8693, 740–755. https://doi.org/10.1007/978-3-319-10602-1_48
6. Guo, C., Pleiss, G., Sun, Y., Weinberger, K. Q. (2017). On calibration of modern neural networks. *ICML 2017*, PMLR 70, 1321–1330. https://proceedings.mlr.press/v70/guo17a.html
7. Angelopoulos, A. N., Bates, S. (2021). A gentle introduction to conformal prediction and distribution-free uncertainty quantification. arXiv:2107.07511. https://arxiv.org/abs/2107.07511
8. Kalra, N., Paddock, S. M. (2016). Driving to safety: How many miles of driving would it take to demonstrate autonomous vehicle reliability? *Transportation Research Part A*, 94, 182–193. https://doi.org/10.1016/j.tra.2016.09.010
9. ISO 26262:2018 (parts 2, 3, 5, 9), ISO 21448:2022, ISO/SAE 21434:2021, ISO/PAS 8800:2024, ISO 34502:2022, ISO 34503:2023. Geneva: ISO.
10. UL Standards & Engagement (2023). *UL 4600: Standard for Safety for the Evaluation of Autonomous Products*, 3rd ed.
11. Patel, M., Jung, R. (2026). Uncertainty evaluation to support safety of the intended functionality analysis for identifying performance insufficiencies in ML-based LiDAR object detection. *Proc. VEHITS 2026*, 154–167. https://doi.org/10.5220/0014967200004030
12. Patel, M., Jung, R. (2026). Integrating cybersecurity into the AI safety assurance argument: A GSN pattern for AI-based perception components in highly automated driving. *SAFECOMP 2026 Workshops*, LNCS, 763–779. https://doi.org/10.1007/978-3-032-35506-5_55

---
**Navigation:** ← [04 Safety case with GSN](04_safety_case_gsn.ipynb) · [Module index](README.md) · [Course overview](../README.md) →